In [ ]:
%load_ext autoreload
%autoreload 2

import torch
import numpy as np
import tqdm
import matplotlib.pyplot as plt

from utils.model_definitions.mteb_automodel_wrapper import AutoModelWrapper, ModelSpecifications

from utils.misc.metric_utils import (
    compute_per_forward_pass,
    compute_on_concatenated_passes,
    metric_name_to_function,
    EvaluationMetricSpecifications,
    entropy_normalization
)
from utils.misc.model_dataloader_utils import (
    model_name_to_sizes, 
    get_model_path, 
    get_dataloader, 
    get_augmentation_collated_dataloader,
)

In [33]:
def compute_sentence_entropies(model, dataloader, granularity='sentence', alpha=1):
    """
    Compute the entropy of each sentence in the dataloader.
    """
    compute_func_kwargs = {
        'alpha': alpha,
        'normalizations': ['maxEntropy', 'raw', 'length']
    }
    compute_function =  metric_name_to_function['entropy']
    forward_pass_func = compute_per_forward_pass if granularity == 'sentence' else compute_on_concatenated_passes

    results = forward_pass_func(model, dataloader, compute_function, should_average_over_layers=False, **compute_func_kwargs)
    return results

# Datasets with Varying Degrees of Intensity

In [48]:
import random

def make_wikitext_with_randomness(tokenizer, old_dataloader, random_strength=0.1):
    """
    Make tokens in old_dataloader random with probability proportional to random_strength
    """
    assert 0 <= random_strength <= 1, "random_strength must be between 0 and 1"

    new_dataloader = []

    for sample in old_dataloader:
        input_ids = sample['input_ids'].tolist()[0]
        new_input_ids = []
        for token in input_ids:
            if random.random() < random_strength:
                new_input_ids.append(random.randint(20, len(tokenizer.vocab) - 1))
            else:
                new_input_ids.append(token)

        new_dataloader.append({'input_ids': torch.tensor(new_input_ids).unsqueeze(0), 'attention_mask': sample['attention_mask']})

    return new_dataloader

def make_completely_random_of_length(tokenizer, num_samples, length=10):
    """
    Make a sentence of a given length with random tokens
    """
    new_dataloader = []
    for idx in range(num_samples):
        print(f"generating sample {idx}")
        input_ids = torch.randint(20, len(tokenizer.vocab), (length,)).tolist()
        attention_mask = [1] * length
        new_dataloader.append({'input_ids': torch.tensor(input_ids).unsqueeze(0), 'attention_mask': torch.tensor(attention_mask).unsqueeze(0)})
    return new_dataloader

def make_wikitext_with_repetition(old_dataloader, repetition_strength=0.1):
    """
    Select a random token. Make repetition_strength percent of tokens in the sentence equal to that token.
    """
    assert 0 <= repetition_strength <= 1, "repetition_strength must be between 0 and 1"

    new_dataloader = []

    for sample in old_dataloader:
        input_ids = sample['input_ids'].tolist()[0]
        chosen_token = random.choice(input_ids)
        new_input_ids = []
        for token in input_ids:
            if random.random() < repetition_strength:
                new_input_ids.append(chosen_token)
            else:
                new_input_ids.append(token)
        new_dataloader.append({'input_ids': torch.tensor(new_input_ids).unsqueeze(0), 'attention_mask': sample['attention_mask']})

    return new_dataloader

In [ ]:
model_specs = ModelSpecifications(
    model_family="Llama3",
    model_size="8B",
    revision="main"
)
model = AutoModelWrapper(model_specs, device_map="auto")

In [ ]:
import pickle


dataloader = get_dataloader(model.tokenizer, "wikitext", split="train", num_samples=5000, min_length=30)
randomness_levels = np.linspace(0, 1, 10)
results = {}
for randomness_level in randomness_levels:
    try:
        new_dataloader = make_wikitext_with_randomness(model.tokenizer, dataloader, randomness_level)
        trainset_entropies = compute_sentence_entropies(model, new_dataloader, alpha=1)
        results[randomness_level] = trainset_entropies
    except Exception as e:
        print(f"Error at randomness level {randomness_level}: {e}")

with open(f'{model_specs.model_family}_randomness.pkl', 'wb') as f:
    pickle.dump(results, f)

In [ ]:
dataloader = get_dataloader(model.tokenizer, "wikitext", split="train", num_samples=5000, min_length=30)
repetition_levels = np.linspace(0, 0.9, 9)
results = {}
for repetition_level in repetition_levels:
    try:
        new_dataloader = make_wikitext_with_repetition(model.tokenizer, dataloader, repetition_level)
        trainset_entropies = compute_sentence_entropies(model, new_dataloader, alpha=1)
        results[repetition_level] = trainset_entropies
    except Exception as e:
        print(f"Error at repetition level {repetition_level}: {e}")

with open(f'{model_specs.model_family}_repetition.pkl', 'wb') as f:
    pickle.dump(results, f)

In [ ]:
lengths = np.logspace(6, 11, 6, base=2, dtype=int)
results = {}
for length in lengths:
    try:    
        print(f"doing length {length}")
        new_dataloader = make_completely_random_of_length(model.tokenizer, num_samples=1000, length=length)
        trainset_entropies = compute_sentence_entropies(model, new_dataloader, alpha=1)
        results[length] = trainset_entropies
    except Exception as e:
        print(f"Error: {e}")
        continue

with open(f'{model_specs.model_family}_random_of_differing_lengths.pkl', 'wb') as f:
    pickle.dump(results, f)

# Old

## Lets take a look at how entropy evolves as context length increases

Here I adjusting the context length of the dataset by truncating each sentence so that its resulting length is a fixed percentage of its full length. Specifically I use percentages ranging from 0.1 to 1.0, where at 0.1 the sentence is truncated to the first 10% of words, and at 1.0 the sentence is fully complete.

The goal here is to see how information content / entropy changes as we increase the length of sentences.

In [ ]:
context_length_ratios = np.linspace(0.1, 1.0, 5)
context_length_entropies = []

for context_length_ratio in context_length_ratios:
    print(f"doing context length of {context_length_ratio}")
    dataset = get_dataloader(tokenizer, "wikitext", split="train", context_length_ratio=context_length_ratio, min_length=50, num_samples=25000)
    context_length_entropies.append(compute_entropies_for_each_sentence(model, dataset, alpha=1))

In [ ]:
# plot the average entropy vs context length increases
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
for context_length_ratio, context_length_entropy in zip(context_length_ratios, context_length_entropies):
    ax[0].hist(context_length_entropy['logN_normalized'], bins=50, density=True, alpha=0.5, label=f"ratio={context_length_ratio:.2f}")
    ax[1].hist(context_length_entropy['unnormalized'], bins=50, density=True, alpha=0.5, label=f"ratio={context_length_ratio:.2f}")

ax[0].set_title('logN normalized entropy')
ax[0].set_xlabel("Entropy")
ax[0].set_ylabel("Density")
ax[0].legend()

ax[1].set_title('unnormalized entropy')
ax[1].set_xlabel("Entropy")
ax[1].set_ylabel("Density")
ax[1].legend()

The above graph is pretty cool because the left and right sides have different trends. The left side roughly captures "entropy per word", which decreases as sentence lengths increase. The right side captures "total entropy", which increases as sentence lengths increase.

Below, I am plotting the average entropies vs context length

In [ ]:
average_logD_entropies = [np.mean(context_length_entropy['logD_normalized']) for context_length_entropy in context_length_entropies]
average_logN_enropies = [np.mean(context_length_entropy['logN_normalized']) for context_length_entropy in context_length_entropies]
average_unnormalized_entropies = [np.mean(context_length_entropy['unnormalized']) for context_length_entropy in context_length_entropies]

fig, axes = plt.subplots(1, 2, figsize=(8,4))
keys = ['logN_normalized', 'unnormalized']
for ax, key in zip(axes.flatten(), keys):
    average_entropy = [np.mean(context_length_entropy[key]) for context_length_entropy in context_length_entropies]
    ax.plot(context_length_ratios, average_entropy)
    ax.set_title(key)
    ax.set_xlabel("context length ratio")
    ax.set_ylabel("Average Entropy")
    
plt.tight_layout()
plt.show()

## How does one word repeated many times behave?

Here I take the word "buffalo" and repeat it several times. The goal is to see how sentences constructed this way behave compared to regular sentences.

In [11]:
import random

special_word = "serat"
lengths = list(range(2, 100))
entropies = []

entropy_dict = {
        'unnormalized': [],
        'logN_normalized': [],
        'logD_normalized': [],
        'logNlogD_normalized': [],
        'lengths': []
    }

for length in lengths:
    input_string = ' '.join([special_word] * length)

    tokenized_string= tokenizer(input_string, truncation=False, return_tensors='pt')
    tokenized_string = {k: v.to(device) for k, v in tokenized_string.items()}

    with torch.no_grad():
        outputs = model(**tokenized_string)
        hidden_states = outputs.hidden_states
        N, D = hidden_states[0].shape[1:]

        last_hidden_state = normalize(outputs.last_hidden_state.squeeze())
        if N > D:
            cov = last_hidden_state.T @ last_hidden_state
        else:
            cov = (last_hidden_state @ last_hidden_state.T)
        cov /= torch.trace(cov)
        entropy = itl.matrixAlphaEntropy(cov, alpha=1)

        entropy_dict['unnormalized'].append(entropy.item())
        entropy_dict['logN_normalized'].append(entropy.item() / math.log(N))
        entropy_dict['logD_normalized'].append(entropy.item() / math.log(D))
        entropy_dict['logNlogD_normalized'].append(entropy.item() / (math.log(N)*math.log(D)))
        entropy_dict['lengths'].append(N)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(8, 3.5))
keys = ['logN_normalized', 'unnormalized']
for ax, key in zip(axes.flatten(), keys):
    ax.plot(lengths, entropy_dict[key])
    ax.set_title(key)
    ax.set_xlabel("Number of Repetitions")
    ax.set_ylabel("Entropy")

    # show max point
    max_index = np.argmax(entropy_dict[key])
    ax.scatter(lengths[max_index], entropy_dict[key][max_index], color='red')
    ax.annotate(f"max={max_index+2}, {entropy_dict[key][max_index]:.2f}", (lengths[max_index], entropy_dict[key][max_index]))

fig.suptitle(f"Entropy Behavior of Repeating '{special_word}'")

## Entropy of Sentences with Random Words

In [13]:
import random

lengths = 200
number_of_random_sentences = 2

random_sentences = [[tokenizer.decode(random.randint(100, 30000)) for x in range(lengths)] for _ in range(number_of_random_sentences)]
random_sentences[1] = "Elden Ring contains crafting mechanics; the creation of items requires materials. Recipes, which are required for the crafting of items, can be found inside collectibles called cookbooks, which are scattered throughout the world. Materials can be collected by defeating enemies, exploring the game's world, or by trading with merchant NPCs. Crafted items include poison darts, exploding pots, and consumables that temporarily increase the player's combat strength.[17][18] Similar to the Dark Souls games, the player can summon friendly NPCs called spirits to fight enemies.[19] Summoning each type of spirit requires its equivalent Spirit Ash; different types of Spirit Ashes can be discovered as the player explores the game world. Spirits can only be summoned near structures called Rebirth Monuments, which are primarily found in large areas and inside boss fight arenas.[20] ".split(" ")
entropy_dict = [{
        'unnormalized': [],
        'logN_normalized': [],
        'logD_normalized': [],
        'logNlogD_normalized': [],
        'lengths': []
    } for _ in range(number_of_random_sentences)]

num_tokens = [[] for _ in range(number_of_random_sentences)]

for i in range(number_of_random_sentences):
    for length in range(2, lengths):
        input_string = ' '.join(random_sentences[i][:length])

        tokenized_string= tokenizer(input_string, truncation=False, return_tensors='pt')
        tokenized_string = {k: v.to(device) for k, v in tokenized_string.items()}

        with torch.no_grad():
            outputs = model(**tokenized_string)
            hidden_states = outputs.hidden_states
            N, D = hidden_states[0].shape[1:]
            num_tokens[i].append(N)
            
            last_hidden_state = normalize(outputs.last_hidden_state.squeeze())
            if N > D:
                cov = last_hidden_state.T @ last_hidden_state
            else:
                cov = (last_hidden_state @ last_hidden_state.T)
            cov /= torch.trace(cov)
            entropy = itl.matrixAlphaEntropy(cov, alpha=1)

            entropy_dict[i]['unnormalized'].append(entropy.item())
            entropy_dict[i]['logN_normalized'].append(entropy.item() / math.log(N))
            entropy_dict[i]['logD_normalized'].append(entropy.item() / math.log(D))
            entropy_dict[i]['logNlogD_normalized'].append(entropy.item() / (math.log(N)*math.log(D)))
            entropy_dict[i]['lengths'].append(N)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(8, 3.7))
keys = ['logN_normalized', 'unnormalized']
for ax, key in zip(axes.flatten(), keys):
    for i in range(number_of_random_sentences):
        ax.plot(list(range(2, lengths)), entropy_dict[i][key])
    
    
    ax.set_title(key)
    ax.set_xlabel("Sentence Length")
    ax.set_ylabel("Entropy")

axes[0].set_ylim(0, 1)
axes[1].plot( list(range(2, 200)), np.log(num_tokens[0]), color='black', linestyle='--', label='log(N)')
axes[1].legend()
fig.suptitle(f"Entropy Behavior of Random Sentence")

The above graph is interesting because the overall entropy here is going down. This is a little strange, but kind of makes sense because increasing the number of repetitions doesn't add information

## Is there an eigenvector that encodes repetition?

Here I repeat the word "buffalo" 10 times. I embed the sentence and compute the covariance eigenvectors

In [15]:
special_word = "buffalo"

input_string = ' '.join([special_word] * 10)
tokenized_string= tokenizer(input_string, truncation=False, return_tensors='pt')
tokenized_string = {k: v.to(device) for k, v in tokenized_string.items()}

with torch.no_grad():
    outputs = model(**tokenized_string)
    hidden_states = outputs.hidden_states
    N, D = hidden_states[0].shape[1:]

    last_hidden_state = normalize(outputs.last_hidden_state.squeeze())

    cov = (last_hidden_state.T @ last_hidden_state)
    cov /= torch.trace(cov)
    entropy = itl.matrixAlphaEntropy(cov, alpha=1)

    # get eigs and vecs of cov
    eigs, vecs = torch.linalg.eigh(cov)
    eigs, indices = torch.sort(eigs, descending=True)
    vecs = vecs[:, indices].T


    buffalo_length_10_eigenvectors = vecs

So far we have the eigenvectors for the sentence "buffalo" repeated 10 times. Next we will see how those eigenvectors evolve as the number of repetitions increases. We will also see how they look on different words.

In [ ]:
import numpy as np

special_words = ["buffalo", "$user", "giraffe", "vehicle", "buff"]
lengths = list(range(10, 100))
entropies = []

repeated_eigenvals_over_time = np.zeros((len(special_words), D, len(lengths)))

with torch.no_grad():
    for word_idx, word in enumerate(special_words):
        for length in tqdm.tqdm(lengths):
            input_string = ' '.join([word] * length)
            tokenized_string= tokenizer(input_string, truncation=False, return_tensors='pt')
            tokenized_string = {k: v.to(device) for k, v in tokenized_string.items()}


            outputs = model(**tokenized_string)
            hidden_states = outputs.hidden_states
            N, D = hidden_states[0].shape[1:]

            last_hidden_state = normalize(outputs.last_hidden_state.squeeze())

            cov = (last_hidden_state.T @ last_hidden_state)
            cov /= torch.trace(cov)

            for vec_idx, vec in enumerate(buffalo_length_10_eigenvectors):
                # rayleigh quotient
                repeated_eigenvals_over_time[word_idx][vec_idx][length-10] = (vec.T @ cov @ vec) / (vec.T @ vec)

Now we take the top 5 eigenvectors and see how they evolve for our repeated sentences and our dataset sentences as length increases.

In [ ]:
eig_idx_min, eig_idx_max = 0, 5

fig, axes = plt.subplots(1, eig_idx_max-eig_idx_min, figsize=( len(special_words)*8, 8))

for vec_idx in range(eig_idx_min, eig_idx_max):
    for word_idx, word in enumerate(special_words):
        axes[vec_idx-eig_idx_min].plot(lengths, repeated_eigenvals_over_time[word_idx][vec_idx], label=word)
    axes[vec_idx-eig_idx_min].legend()
    axes[vec_idx-eig_idx_min].set_title(f"eigenvector {vec_idx+1}")

    # all_average_entropies = [np.mean(x) for x in all_vals]
    # axes[1].scatter(lengths, all_average_entropies)

axes[0].set_xlabel("Sentence Length")
axes[0].set_ylabel("Eigenvalue")

plt.show()

## Lets take a look at how entropy changes as model size is increased

In [ ]:
logN_sample_entropies = {k: [] for k in EleutherAI_sizes}
logD_sample_entropies = {k: [] for k in EleutherAI_sizes}
logNlogD_sample_entropies = {k: [] for k in EleutherAI_sizes}
unnormalized_sample_entropies = {k: [] for k in EleutherAI_sizes}

dimensionalities = {k: 0 for k in EleutherAI_sizes}

for model_size in EleutherAI_sizes:
    print("Running ", model_size)
    model_path = get_model_path("EleutherAI", model_size)

    tokenizer = AutoTokenizer.from_pretrained(model_path)
    model = AutoModel.from_pretrained(model_path, output_hidden_states=True).to(device)
    dataloader = get_dataloader(tokenizer, "wikitext", split="train", num_samples=10000)

    entropies = compute_entropies_for_each_sentence(model, dataloader, alpha=1)
    logN_sample_entropies[model_size] = entropies['logN_normalized']
    logD_sample_entropies[model_size] = entropies['logD_normalized']
    logNlogD_sample_entropies[model_size] = entropies['logNlogD_normalized']
    unnormalized_sample_entropies[model_size] = entropies['unnormalized']

    # get dimensionality of model
    with torch.no_grad():
        outputs = model(torch.zeros(1, 1).long().to(device))
        D = outputs.last_hidden_state.shape[-1]
        dimensionalities[model_size] = D

In [ ]:
logN_average_entropies = [np.mean(logN_sample_entropies[k]) for k in logN_sample_entropies.keys()]
logD_average_entropies = [np.mean(logD_sample_entropies[k]) for k in logD_sample_entropies.keys()]
logNlogD_average_entropies = [np.mean(logNlogD_sample_entropies[k]) for k in logNlogD_sample_entropies.keys()]
unnormalized_average_entropies = [np.mean(unnormalized_sample_entropies[k]) for k in unnormalized_sample_entropies.keys()]

print(unnormalized_sample_entropies)
model_params = [int(x[:-1])*10**6 for x in ['14m', '70m', '160m', '410m', '1000m', '1400m', '2800m',]]#if x[-1]=='m' else x[:-1]*10**7]

fig, axes = plt.subplots(2, 2, figsize=(10,10))
print(np.mean(unnormalized_sample_entropies['14m']))

axes[0][0].plot(model_params, unnormalized_average_entropies, label="entropy")
axes[0][0].plot(model_params, np.log(list(dimensionalities.values())), label="log(D)")
axes[0][0].set_title("unnormalized")
axes[0][0].set_xlabel("# of Model Parameters")
axes[0][0].set_ylabel("Entropy")
axes[0][0].legend()

axes[0][1].plot(model_params, logN_average_entropies)
axes[0][1].set_title("logN normalized")
axes[0][1].set_xlabel("# of Model Parameters")
axes[0][1].set_ylabel("Entropy")

axes[1][0].plot(model_params, logD_average_entropies)
axes[1][0].set_title("logD normalized")
axes[1][0].set_xlabel("# of Model Parameters")
axes[1][0].set_ylabel("Entropy")

axes[1][1].plot(model_params, logNlogD_average_entropies)
axes[1][1].set_title("logNlogD normalized")
axes[1][1].set_xlabel("# of Model Parameters")
axes[1][1].set_ylabel("Entropy")

plt.show()